### Imports

In [2]:
import os
import pickle

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from dotenv import load_dotenv
load_dotenv()

True

### Load Documents

In [4]:
with open("../data/processed_data/03_rag_documents.pkl", "rb") as f:
    documents = pickle.load(f)

print("Total Documents:", len(documents))

Total Documents: 2331


In [5]:
documents[0]

Document(metadata={'document_id': 1, 'category': 'Retail Banking', 'original_category': 'accounts', 'question': 'What are the documents required for opening a Current Account of a sole proprietorship firm'}, page_content='Category: Retail Banking\n\nQuestion: What are the documents required for opening a Current Account of a sole proprietorship firm\n\nAnswer: Following documents are required to open a Current Account of a sole proprietorship entity: Proof of existence in the name of firm Proof of address in the name of firm KYC of the proprietor Any two of the below listed documents shall be obtained for establishing proof of existence. Registration certificate/license issued by Municipal authorities such as Shop & Establishment Certificate/Trade License CST/VAT/Service Tax Certificate or Letter Of Registration for CST/VAT/Service Tax Certificate/Registration document issued by Professional Tax authorities Valid Business License or Certificate Of Registration issued by State/Central G

### Load Embedding Model

In [7]:
def load_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

    return embeddings

embeddings = load_embeddings()
print("Embedding Model Loaded")

Embedding Model Loaded


#### Test Embedding Generation

In [8]:
sample_embedding = embeddings.embed_query("How can I reset my debit card PIN?")

len(sample_embedding)

384

### Create FAISS Vector Database

In [9]:
vectorstore = FAISS.from_documents(documents, embeddings)

print(type(vectorstore))

<class 'langchain_community.vectorstores.faiss.FAISS'>


#### Similarity Search Test

In [10]:
query = "How do I block my ATM card?"

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print("=" * 80)
    print(f"Result {i}")
    print(doc.page_content[:500])

Result 1
Category: Customer Support

Question: How do I block my debit card?

Answer: Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly.
Result 2
Category: Cards & Payments

Question: What is the procedure to follow if my Debit Card PIN is blocked

Answer: Please note that if you enter an incorrect PIN three times in the ATM, your access gets blocked for security reasons. It gets activated after 24 hours. Kindly use your Debit / ATM Card at the ATM after 24 hours with the same PIN available with you. If your account still remains inaccessible, please apply for new PIN. You can apply for regeneration of your ATM / Debit PIN in following wa
Result 3
Category: Digital & Security

Question: If my account is blocked, how do I unblock it

Answer: If your account gets blocked, please contact your nearest Customer Call Centre (for Credit Cards) and PhoneBanking centre (for Debit Cards).


#### Similarity Search With Score

In [12]:
results = vectorstore.similarity_search_with_score(query, k=3)

for doc, score in results:
    print("=" * 80)
    print("Score:", score)
    print(doc.page_content[:300])

Score: 0.6593218
Category: Customer Support

Question: How do I block my debit card?

Answer: Call your bank's 24x7 helpline, use the mobile banking app, or send an SMS as instructed by your bank. Your card will be blocked instantly.
Score: 0.7979048
Category: Cards & Payments

Question: What is the procedure to follow if my Debit Card PIN is blocked

Answer: Please note that if you enter an incorrect PIN three times in the ATM, your access gets blocked for security reasons. It gets activated after 24 hours. Kindly use your Debit / ATM Card at t
Score: 0.8507578
Category: Digital & Security

Question: If my account is blocked, how do I unblock it

Answer: If your account gets blocked, please contact your nearest Customer Call Centre (for Credit Cards) and PhoneBanking centre (for Debit Cards).


#### Category-wise Retrieval Test

##### Retail Banking Query

In [13]:
query = "How to open savings account?"

results = vectorstore.similarity_search(query, k=3)

for doc in results:
    print(doc.metadata)

{'document_id': 212, 'category': 'Retail Banking', 'original_category': 'accounts', 'question': 'What are the documents required to open a New Savings Account'}
{'document_id': 226, 'category': 'Retail Banking', 'original_category': 'accounts', 'question': "What are the documents required to open a Women's Savings Account"}
{'document_id': 215, 'category': 'Retail Banking', 'original_category': 'accounts', 'question': 'What is the general documentation requirement to open an Institutional Savings Account'}


##### Loan Query

In [14]:
query = "What is home loan EMI?"

results = vectorstore.similarity_search(query, k=3)

for doc in results:
    print(doc.metadata)

{'document_id': 1233, 'category': 'Loans', 'original_category': 'loans', 'question': 'What is EMI'}
{'document_id': 1571, 'category': 'Loans', 'original_category': 'Loans & Interest', 'question': 'What is an EMI?'}
{'document_id': 1651, 'category': 'Loans', 'original_category': 'Loans & Interest', 'question': 'What is an EMI holiday?'}


### Save FAISS Index

In [15]:
os.makedirs("../vectorstore/faiss_index", exist_ok=True)
vectorstore.save_local("../vectorstore/faiss_index")

### Reload Saved FAISS Index

In [16]:
loaded_vectorstore = FAISS.load_local("../vectorstore/faiss_index", embeddings, allow_dangerous_deserialization=True)
print("FAISS Reloaded Successfully")

FAISS Reloaded Successfully


#### Verify Reloaded Index

In [17]:
results = loaded_vectorstore.similarity_search("What is RTGS transfer?", k=3)

for doc in results:
    print(doc.page_content[:300])

Category: Digital & Security

Question: What is RTGS?

Answer: RTGS is Real-Time Gross Settlement. It's for transferring large amounts (above ₹2 lakh) instantly. It works 24x7 and is settled in real time.
Category: Loans

Question: What is RTGS

Answer: RTGS is the Real Time Gross Settlement which is used for the transfer of amounts of Rs. 1,00,000 or more from one bank account to another bank account (any bank located anywhere in India if attached with an Internet system) at very minimal charges. Th
Category: Cards & Payments

Question: What is RTGS Funds Transfer

Answer: 'RTGS' stands for 'Real Time Gross Settlement'. The RTGS system is a funds transfer mechanism where transfer of money takes place from one bank to another on a 'real time' and on a 'gross' basis. This is the fastest possible 


### Save Embedding Metadata

In [18]:
embedding_config = {
    "model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "vector_dimensions": 384,
    "total_documents": len(documents)
}

with open("../models/rag/embedding_config.pkl", "wb") as f:
    pickle.dump(embedding_config, f)

## Key Insights

### Documents Prepared

- Loaded banking documents from document preparation stage.
- Each document contains:
  - Question
  - Answer
  - Category metadata

### Embedding Model

- Model:
  sentence-transformers/all-MiniLM-L6-v2

- Vector Dimension:
  384

### FAISS Database

- Built successfully using LangChain FAISS.

### Retrieval Performance

- Similar banking questions retrieve highly relevant FAQ entries.
- Category metadata is preserved for downstream filtering.

### Persistence

- FAISS index saved locally.
- Successfully reloaded and verified.

### Next Step

Proceed to:

08_rag_pipeline.ipynb

to combine:

User Query
→ Retriever
→ Context Builder
→ Llama-3.3-70B
→ Banking Answer